In [1]:
def micro_f1_score(gold_standard, system_output):
  """
  Evaluates a term extraction system's performance using Precision, Recall,
  and F1 score based on individual term matching (micro-average).

  Args:
    gold_standard: A list of lists, where each inner list contains the
                   gold standard terms for an item.
    system_output: A list of lists, where each inner list contains the
                   terms extracted by the system for the corresponding item.

  Returns:
    A tuple containing the Precision, Recall, and F1 score.
  """
  total_true_positives = 0
  total_false_positives = 0
  total_false_negatives = 0

  # Iterate through each item's gold standard and system output terms
  for gold, system in zip(gold_standard, system_output):
    # Convert to sets for efficient comparison
    gold_set = set(gold)
    system_set = set(system)

    # Calculate True Positives, False Positives, and False Negatives for the current item
    true_positives = len(gold_set.intersection(system_set))
    false_positives = len(system_set - gold_set)
    false_negatives = len(gold_set - system_set)

    # Accumulate totals across all items
    total_true_positives += true_positives
    total_false_positives += false_positives
    total_false_negatives += false_negatives

  # Calculate Precision, Recall, and F1 score (micro-average)
  precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
  recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
  f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

  return precision, recall, f1

In [2]:
def type_f1_score(gold_standard, system_output):
  """
  Evaluates a term extraction system's performance using Type Precision,
  Type Recall, and Type F1 score based on the set of unique terms extracted
  at least once across the entire dataset.

  Args:
    gold_standard: A list of lists, where each inner list contains the
                   gold standard terms for an item.
    system_output: A list of lists, where each inner list contains the
                   terms extracted by the system for the corresponding item.

  Returns:
    A tuple containing the Type Precision, Type Recall, and Type F1 score.
  """

  # Get the set of all unique gold standard terms across the dataset
  all_gold_terms = set()
  for item_terms in gold_standard:
    all_gold_terms.update(item_terms)

  # Get the set of all unique system extracted terms across the dataset
  all_system_terms = set()
  for item_terms in system_output:
    all_system_terms.update(item_terms)

  # Calculate True Positives (terms present in both sets)
  type_true_positives = len(all_gold_terms.intersection(all_system_terms))

  # Calculate False Positives (terms in system output but not in gold standard)
  type_false_positives = len(all_system_terms - all_gold_terms)

  # Calculate False Negatives (terms in gold standard but not in system output)
  type_false_negatives = len(all_gold_terms - all_system_terms)

  # Calculate Type Precision, Type Recall, and Type F1 score
  type_precision = type_true_positives / (type_true_positives + type_false_positives) if (type_true_positives + type_false_positives) > 0 else 0
  type_recall = type_true_positives / (type_true_positives + type_false_negatives) if (type_true_positives + type_false_negatives) > 0 else 0
  type_f1 = 2 * (type_precision * type_recall) / (type_precision + type_recall) if (type_precision + type_recall) > 0 else 0

  return type_precision, type_recall, type_f1

In [3]:
# Get file paths for system output and gold standard from user input
SYSTEM_OUTPUT_PATH = "../src/predictions/subtaskA_a_vanilla_preds_filtered.json"
GOLD_STANDARD_PATH = "../data/taskA/test.json"

In [4]:
import json 

# Load data from the specified file paths
system_output_dict = json.load(open(SYSTEM_OUTPUT_PATH, 'r', encoding='utf-8'))
gold_standard_dict = json.load(open(GOLD_STANDARD_PATH, 'r', encoding='utf-8'))

# Extract the term lists from the loaded data dictionaries
gold_standard = list(content['terms'] for content in gold_standard_dict.values())
system_output = list(content['terms'] for content in system_output_dict.values())

# Calculate and print micro-averaged precision, recall, and F1 score
precision, recall, f1 = micro_f1_score(gold_standard, system_output)
print("micro-averaged Precision:", round(precision, 3))
print("micro-averaged Recall:", round(recall, 3))
print("micro-averaged F1 score:", round(f1, 3))

# Calculate and print Type Precision, Type Recall, and Type F1 score
type_precision, type_recall, type_f1 = type_f1_score(gold_standard, system_output)
print("\nType Precision:", round(type_precision, 3))
print("Type Recall:", round(type_recall, 3))
print("Type F1 score:", round(type_f1, 3))

micro-averaged Precision: 0.326
micro-averaged Recall: 0.479
micro-averaged F1 score: 0.388

Type Precision: 0.625
Type Recall: 0.179
Type F1 score: 0.278


In [7]:
import os
import glob
import pandas as pd

# Path to predictions directory and baseline
predictions_dir = "../src/predictions/"
gold_standard_file = "../data/taskA/test.json"

# Get all JSON files in the predictions directory
prediction_files = glob.glob(os.path.join(predictions_dir, "*.json"))

# Load gold standard once
gold_standard_dict = json.load(open(gold_standard_file, 'r', encoding='utf-8'))
gold_standard = list(content['terms'] for content in gold_standard_dict.values())

gold_standard = gold_standard[:20]

# Store results
results = []

# Evaluate each prediction file
for pred_file in prediction_files:
    if not pred_file.endswith(".json"): 
        continue
    file_name = os.path.basename(pred_file)
    print(f"Evaluating {file_name}...")
    
    try:
        # Load system output
        system_output_dict = json.load(open(pred_file, 'r', encoding='utf-8'))
        system_output = list(content['terms'] for content in system_output_dict.values())
        
        # Calculate micro-averaged metrics
        precision, recall, f1 = micro_f1_score(gold_standard, system_output)
        
        # Calculate type metrics
        type_precision, type_recall, type_f1 = type_f1_score(gold_standard, system_output)
        
        # Store results with cleaned file name
        clean_name = file_name.replace("_preds", "").replace(".json", "")
        results.append({
            'file_name': clean_name,
            'micro_precision': round(precision, 3),
            'micro_recall': round(recall, 3),
            'micro_f1': round(f1, 3),
            'type_precision': round(type_precision, 3),
            'type_recall': round(type_recall, 3),
            'type_f1': round(type_f1, 3)
        })
    except Exception as e:
        print(f"Error evaluating {file_name}: {e}")

# Create DataFrame and sort by micro F1 score (descending)
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('micro_f1', ascending=False).reset_index(drop=True)
#results_df = results_df.sort_values('type_f1', ascending=False).reset_index(drop=True)

# Display the table
print("\n" + "="*100)
print("PARTIAL EVALUATION RESULTS")
print("="*100)
print(results_df.to_string(index=False))

# Save to CSV
output_csv = "subtask_a_partial_evaluation_results.csv"
results_df.to_csv(output_csv, index=False)
print(f"\nResults saved to {output_csv}")

Evaluating subtaskA_a_vanilla_preds.json...
Evaluating subtaskA_a_vanilla_preds_filtered.json...
Evaluating subtaskA_bert-base-uncased_preds.json...
Evaluating subtaskA_gemma3-finetuned_preds.json...
Evaluating subtaskA_gpt-oss-120b_preds.json...
Evaluating subtaskA_gpt-oss-20b_preds.json...
Evaluating subtaskA_nltk_baseline_preds.json...
Evaluating subtaskA_nltk_trained_preds.json...
Evaluating subtaskA_qwen2.5-1.5b-instruct_preds.json...
Evaluating subtaskA_spacy_baseline_preds.json...
Evaluating subtaskA_spacy_trained_preds.json...

PARTIAL EVALUATION RESULTS
                     file_name  micro_precision  micro_recall  micro_f1  type_precision  type_recall  type_f1
        subtaskA_spacy_trained            0.760         0.745     0.753           0.059        0.751    0.109
    subtaskA_bert-base-uncased            0.590         0.524     0.555           0.044        0.600    0.081
       subtaskA_spacy_baseline            0.470         0.598     0.527           0.102        0.520 